In [62]:
#%pip install -q -U requests numpy pandas matplotlib transformers accelerate torch

In [63]:
import ast
import json
import re
import time
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional

import numpy as np
import pandas as pd
import requests

np.set_printoptions(precision=5, suppress=True)
pd.set_option("display.max_colwidth", 140)

In [64]:
class ExampleMCP:
    """
    A minimal, in-memory MCP-like registry.
    It is not a full MCP server. It mirrors the core idea: register tools,
    list tools, and call tools through a structured interface.
    """

    def __init__(self, name: str = "example-agent-tools"):
        self.name = name
        self.tools: Dict[str, Callable[..., Any]] = {}

    def tool(self, fn: Callable[..., Any]) -> Callable[..., Any]:
        self.tools[fn.__name__] = fn
        return fn

    def list_tools(self) -> List[Dict[str, str]]:
        return [
            {"name": name, "description": (fn.__doc__ or "").strip()}
            for name, fn in self.tools.items()
        ]

    def call_tool(self, name: str, arguments: Dict[str, Any]) -> Any:
        if name not in self.tools:
            return {"error": f"Unknown MCP tool: {name}"}
        try:
            return self.tools[name](**arguments)
        except Exception as exc:
            return {"error": f"Tool execution failed: {exc}"}


mcp = ExampleMCP()

In [65]:
@dataclass
class ToolResult:
    tool: str
    ok: bool
    data: Any = None
    error: str = ""
    latency_ms: float = 0.0
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class ToolSpec:
    name: str
    description: str
    input_schema: Dict[str, Any]
    fn: Callable[..., Any]
    risk_level: str = "low"


TOOLS: Dict[str, ToolSpec] = {}


def register_tool(spec: ToolSpec) -> None:
    if spec.name in TOOLS:
        raise ValueError(f"Tool already registered: {spec.name}")
    TOOLS[spec.name] = spec


def call_tool(name: str, arguments: Dict[str, Any]) -> ToolResult:
    """Validate and execute one named tool."""
    start = time.perf_counter()

    if name not in TOOLS:
        return ToolResult(tool=name, ok=False, error=f"Unknown tool: {name}")

    spec = TOOLS[name]
    required = spec.input_schema.get("required", [])
    missing = [key for key in required if key not in arguments]
    if missing:
        return ToolResult(tool=name, ok=False, error=f"Missing required arguments: {missing}")

    try:
        data = spec.fn(**arguments)
        ok = not (isinstance(data, dict) and "error" in data)
        return ToolResult(
            tool=name,
            ok=ok,
            data=data if ok else None,
            error="" if ok else data.get("error", "Tool returned an error."),
            latency_ms=(time.perf_counter() - start) * 1000,
            metadata={"risk_level": spec.risk_level, "arguments": arguments},
        )
    except Exception as exc:
        return ToolResult(
            tool=name,
            ok=False,
            error=str(exc),
            latency_ms=(time.perf_counter() - start) * 1000,
            metadata={"risk_level": spec.risk_level, "arguments": arguments},
        )


In [66]:
def weather_code_to_text(code: int) -> str:
    """Convert selected Open-Meteo weather codes to readable labels."""
    mapping = {
        0: "clear sky",
        1: "mainly clear",
        2: "partly cloudy",
        3: "overcast",
        45: "fog",
        48: "depositing rime fog",
        51: "light drizzle",
        53: "moderate drizzle",
        55: "dense drizzle",
        61: "slight rain",
        63: "moderate rain",
        65: "heavy rain",
        71: "slight snow",
        73: "moderate snow",
        75: "heavy snow",
        80: "slight rain showers",
        81: "moderate rain showers",
        82: "violent rain showers",
        95: "thunderstorm",
    }
    return mapping.get(int(code), f"weather code {code}")


def _get_geo_data(city: str):
    """Resolve city name to coordinates using Open-Meteo geocoding."""
    url = "https://geocoding-api.open-meteo.com/v1/search"
    try:
        res = requests.get(
            url,
            params={"name": city, "count": 1, "language": "en", "format": "json"},
            timeout=10,
        )
        res.raise_for_status()
        data = res.json()
        if not data.get("results"):
            return {"error": f"Location '{city}' not found."}
        return data["results"][0]
    except Exception as e:
        return {"error": str(e)}


@mcp.tool
def get_weather(city: str) -> dict:
    """
    Gets current weather for a city using Open-Meteo.

    Args:
        city: city name, for example "Vienna".

    Returns:
        A dictionary with location, temperature, humidity, condition_code, condition, and wind.
    """
    geo = _get_geo_data(city)
    if isinstance(geo, dict) and "error" in geo:
        return geo

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": geo["latitude"],
        "longitude": geo["longitude"],
        "current": "temperature_2m,relative_humidity_2m,weather_code,wind_speed_10m",
        "current_units": "auto",
    }
    try:
        res = requests.get(url, params=params, timeout=10)
        res.raise_for_status()
        data = res.json()
        curr = data["current"]
        units = data["current_units"]
        code = curr["weather_code"]
        return {
            "location": f"{geo['name']}, {geo['country']}",
            "temperature": f"{curr['temperature_2m']} {units['temperature_2m']}",
            "humidity": f"{curr['relative_humidity_2m']} {units['relative_humidity_2m']}",
            "condition_code": code,
            "condition": weather_code_to_text(code),
            "wind": f"{curr['wind_speed_10m']} {units['wind_speed_10m']}",
        }
    except Exception as e:
        return {"error": f"Weather Error: {str(e)}"}


register_tool(
    ToolSpec(
        name="get_weather",
        description="Get current weather for a city using Open-Meteo.",
        input_schema={
            "type": "object",
            "properties": {"city": {"type": "string", "description": "City name"}},
            "required": ["city"],
        },
        fn=get_weather,
        risk_level="low",
    )
)

In [67]:
@mcp.tool
def invert_matrix(matrix: List[List[float]], decimals: int = 6, condition_threshold: float = 1e12) -> Dict[str, Any]:
    """
    Invert a square numeric matrix.

    Args:
        matrix: square matrix as a nested Python list.
        decimals: number of decimals used for the displayed inverse.
        condition_threshold: reject very ill-conditioned matrices.

    Returns:
        determinant, condition_number, inverse, and residual_max_error.
    """
    arr = np.array(matrix, dtype=float)

    if arr.ndim != 2:
        return {"error": "Input must be a two-dimensional matrix."}
    if arr.shape[0] != arr.shape[1]:
        return {"error": f"Matrix must be square, got shape {arr.shape}."}
    if arr.shape[0] < 2:
        return {"error": "Use at least a 2x2 matrix."}

    det = float(np.linalg.det(arr))
    if abs(det) < 1e-12:
        return {"error": "Matrix is singular or nearly singular; inverse is not reliable."}

    condition_number = float(np.linalg.cond(arr))
    if condition_number > condition_threshold:
        return {"error": f"Matrix is ill-conditioned: condition number {condition_number:.3e}."}

    inv = np.linalg.inv(arr)
    residual = arr @ inv - np.eye(arr.shape[0])
    return {
        "shape": list(arr.shape),
        "determinant": round(det, decimals),
        "condition_number": round(condition_number, decimals),
        "inverse": np.round(inv, decimals).tolist(),
        "residual_max_error": float(np.max(np.abs(residual))),
    }


register_tool(
    ToolSpec(
        name="invert_matrix",
        description="Invert a square numeric matrix and return determinant, condition number, inverse, and residual error.",
        input_schema={
            "type": "object",
            "properties": {
                "matrix": {"type": "array", "description": "Square matrix as nested lists"},
                "decimals": {"type": "integer", "description": "Number of decimals"},
            },
            "required": ["matrix"],
        },
        fn=invert_matrix,
        risk_level="low",
    )
)

DDGS Tool

In [68]:
#import argparse
from ddgs import DDGS

In [69]:
@mcp.tool
def search_web(query: str, max_results: int = 5) -> dict:
    """
    Search the web using DDGS.

    Args:
        query: search query
        max_results: number of results

    Returns:
        Search results or error.
    """

    try:
        with DDGS(timeout=10) as ddgs:
            results = list(
                ddgs.text(
                    query,
                    region="wt-wt",
                    safesearch="moderate",
                    max_results=max_results,
                )
            )

        if not results:
            return {"error": "No search results found."}

        return {
            "query": query,
            "results": results
        }

    except Exception as e:
        return {"error": f"Search error: {e}"}

register_tool(
    ToolSpec(
        name="search_web",
        description="Search the web using DuckDuckGo Search.",
        input_schema={
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query"
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results"
                }
            },
            "required": ["query"],
        },
        fn=search_web,
        risk_level="low",
    )
)

In [70]:
import requests

@mcp.tool

def convert_currency(
    amount: float,
    from_currency: str,
    to_currency: str
):
    try:
        url = (
            f"https://api.frankfurter.dev/v2/rate/"
            f"{from_currency.upper()}/{to_currency.upper()}"
        )

        r = requests.get(url, timeout=10)
        r.raise_for_status()

        data = r.json()

        rate = float(data["rate"])
        converted = round(amount * rate, 2)

        return {
            "amount": amount,
            "from": from_currency.upper(),
            "to": to_currency.upper(),
            "rate": rate,
            "converted": converted,
        }

    except Exception as e:
        return {"error": str(e)}

register_tool(
    ToolSpec(
        name="convert_currency",
        description="Convert money using Frankfurter exchange rates.",
        input_schema={
            "type": "object",
            "properties": {
                "amount": {"type": "number"},
                "from_currency": {"type": "string"},
                "to_currency": {"type": "string"},
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
        fn=convert_currency,
        risk_level="low",
    )
)

In [71]:
convert_currency(100, "EUR", "USD")

{'error': "HTTPSConnectionPool(host='api.frankfurter.dev', port=443): Read timed out. (read timeout=10)"}

In [72]:
@mcp.tool
def calculate(expression):
    try:
        result = eval(expression, {"__builtins__": {}})
        return {"expression": expression, "result": result}

    except Exception as e:
        return {"error": str(e)}

register_tool(
    ToolSpec(
        name="calculate",
        description="Evaluate simple arithmetic expressions.",
        input_schema={
            "type": "object",
            "properties": {
                "expression": {"type": "string"}
            },
            "required": ["expression"]
        },
        fn=calculate,
        risk_level="low",
    )
)

In [73]:
@mcp.tool
def final_answer(answer: str) -> dict:
    """
    Final answer tool used to terminate the agent loop.

    Args:
        answer: final response to the user

    Returns:
        Final structured output
    """
    return {
        "answer": answer
    }

register_tool(
    ToolSpec(
        name="final_answer",
        description="Use this tool to return the final answer and terminate the agent.",
        input_schema={
            "type": "object",
            "properties": {
                "answer": {
                    "type": "string",
                    "description": "Final answer to the user"
                }
            },
            "required": ["answer"],
        },
        fn=final_answer,
        risk_level="low",
    )
)

In [74]:
print("Internal tool registry:")
for tool in TOOLS.values():
    print(f"- {tool.name}: {tool.description}")

print("\nMCP-style list_tools():")
pd.DataFrame(mcp.list_tools())

Internal tool registry:
- get_weather: Get current weather for a city using Open-Meteo.
- invert_matrix: Invert a square numeric matrix and return determinant, condition number, inverse, and residual error.
- search_web: Search the web using DuckDuckGo Search.
- convert_currency: Convert money using Frankfurter exchange rates.
- calculate: Evaluate simple arithmetic expressions.
- final_answer: Use this tool to return the final answer and terminate the agent.

MCP-style list_tools():


,name,description
0,get_weather,"Gets current weather for a city using Open-Meteo.\n\n Args:\n city: city name, for example ""Vienna"".\n\n Returns:\n ..."
1,invert_matrix,Invert a square numeric matrix.\n\n Args:\n matrix: square matrix as a nested Python list.\n decimals: number of decima...
2,search_web,Search the web using DDGS.\n\n Args:\n query: search query\n max_results: number of results\n\n Returns:\n Se...
3,convert_currency,
4,calculate,
5,final_answer,Final answer tool used to terminate the agent loop.\n\n Args:\n answer: final response to the user\n\n Returns:\n Fi...


In [75]:
MODEL_ID = "Qwen/Qwen3.5-0.8B"

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model_kwargs = {
    "trust_remote_code": True,
    "low_cpu_mem_usage": True,
}

if DEVICE == "cuda":
    model_kwargs.update({"torch_dtype": "auto", "device_map": "auto"})
else:
    model_kwargs.update({"torch_dtype": torch.float32})

try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
except Exception as exc:
    raise RuntimeError(
        f"Could not load required model {MODEL_ID}. "
        "Check the model name, internet access, Hugging Face permissions, and installed transformers version."
    ) from exc

if DEVICE == "cpu":
    model.to("cpu")

model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

def model_device_summary(model) -> str:
    try:
        return str(next(model.parameters()).device)
    except StopIteration:
        return "unknown"

print(f"Loaded required model: {MODEL_ID}")
print(f"Model device: {model_device_summary(model)}")


Loading weights: 100%|██████████| 320/320 [00:00<00:00, 2499.97it/s]


Loaded required model: Qwen/Qwen3.5-0.8B
Model device: cpu


In [76]:
def remove_thinking_blocks(text: str) -> str:
    """Remove common hidden-reasoning style tags from model output if present."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    return text.strip()


def qwen_generate(prompt: str, max_new_tokens: int = 1000) -> str:
    """Generate text with the required Qwen model."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict tool-call planner. "
                "Return only one valid JSON object. "
                "Do not use markdown. Do not write explanations outside JSON."
            ),
        },
        {"role": "user", "content": prompt},
    ]

    if getattr(tokenizer, "chat_template", None):
        try:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
    else:
        text = (
            "System: Return only one valid JSON object. No markdown.\n"
            f"User: {prompt}\nAssistant:"
        )

    inputs = tokenizer(text, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    raw = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return remove_thinking_blocks(raw)

In [ ]:
def extract_first_matrix(text: str) -> Optional[List[List[float]]]:
    """Extract the first nested-list matrix like [[1, 2], [3, 4]] from text."""
    start = text.find("[[")
    if start == -1:
        return None

    depth = 0
    for pos in range(start, len(text)):
        char = text[pos]
        if char == "[":
            depth += 1
        elif char == "]":
            depth -= 1
            if depth == 0:
                candidate = text[start:pos + 1]
                try:
                    value = ast.literal_eval(candidate)
                    arr = np.array(value, dtype=float)
                    if arr.ndim == 2:
                        return value
                except Exception:
                    return None
    return None


def extract_city(text: str) -> Optional[str]:
    """Extract a city from common weather questions."""
    cleaned = text.strip()
    patterns = [
        r"weather\s+in\s+([A-Za-zÀ-ÿ .'-]+)\??",
        r"weather\s+for\s+([A-Za-zÀ-ÿ .'-]+)\??",
        r"actual\s+weather\s+in\s+([A-Za-zÀ-ÿ .'-]+)\??",
        r"current\s+weather\s+in\s+([A-Za-zÀ-ÿ .'-]+)\??",
    ]
    for pattern in patterns:
        match = re.search(pattern, cleaned, flags=re.IGNORECASE)
        if match:
            city = match.group(1).strip(" .?'")
            city = re.split(r"\b(today|now|currently|actual|please)\b", city, flags=re.IGNORECASE)[0].strip()
            if city:
                return city

    # Common example used in this notebook.
    if "vienna" in cleaned.lower():
        return "Vienna"
    return None


def parse_json_from_text(text: str) -> Optional[Dict[str, Any]]:
    """Recover the first JSON object from model output."""
    if not text:
        return None

    try:
        value = json.loads(text)
        return value if isinstance(value, dict) else None
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    try:
        value = json.loads(text[start:end + 1])
        return value if isinstance(value, dict) else None
    except Exception:
        return None


def normalize_model_tool_call(parsed: Optional[Dict[str, Any]], question: str, raw: str) -> Optional[Dict[str, Any]]:
    """Validate and normalize a Qwen tool-call proposal."""
    if not isinstance(parsed, dict):
        return None

    tool = parsed.get("tool")
    if tool not in TOOLS:
        return None

    arguments = parsed.get("arguments")
    if not isinstance(arguments, dict):
        arguments = {}

    if tool == "get_weather":
        city = arguments.get("city") or extract_city(question)
        if not city:
            return None
        arguments = {"city": str(city)}

    elif tool == "invert_matrix":
        matrix = arguments.get("matrix") or extract_first_matrix(question)
        if matrix is None:
            return None
        decimals = int(arguments.get("decimals", 5))
        arguments = {"matrix": matrix, "decimals": decimals}

    elif tool == "search_web":
        query = arguments.get("query")
        if not query:
            query = question
        arguments = {"query": query, "max_results": 5}

    return {
        "tool": tool,
        "arguments": arguments,
        "reason": parsed.get("reason", "Qwen selected this tool."),
        "source": "qwen",
        "model_raw": raw,
    }


def guarded_tool_call_from_question(question: str, raw: str) -> Optional[Dict[str, Any]]:
    """Repair only the tool-call structure after Qwen was invoked.

    This keeps the notebook robust with small local models while ensuring that
    the language model is still part of every routing step.
    """
    matrix = extract_first_matrix(question)
    if matrix is not None and re.search(r"invert|inverse|matrix|solve", question, flags=re.IGNORECASE):
        return {
            "tool": "invert_matrix",
            "arguments": {"matrix": matrix, "decimals": 5},
            "reason": "Qwen was invoked; verifier repaired the matrix tool-call structure from the user question.",
            "source": "qwen_guardrail_repair",
            "model_raw": raw,
        }

    city = extract_city(question)
    if city is not None and re.search(r"weather|temperature|wind|humidity", question, flags=re.IGNORECASE):
        return {
            "tool": "get_weather",
            "arguments": {"city": city},
            "reason": "Qwen was invoked; verifier repaired the weather tool-call structure from the user question.",
            "source": "qwen_guardrail_repair",
            "model_raw": raw,
        }

    general_info_keywords = ["what is","who is","when","where","why","how",
    "capital","population","history","information","news","latest","recent",
    "current","explain","compare","describe"]
    if any(keyword in question.lower() for keyword in general_info_keywords):
        return {
            "tool": "search_web",
            "arguments": {
                "query": question,
                "max_results": 5
            },
            "reason": "Qwen was invoked; verifier repaired the search tool-call structure from the user question.",
            "source": "qwen_guardrail_repair",
            "model_raw": raw,
    }

    return None


def qwen_propose_tool_call(question: str) -> Dict[str, Any]:
    """Ask Qwen to select exactly one registered tool."""
    tool_descriptions = [
        {"name": spec.name, "description": spec.description, "schema": spec.input_schema}
        for spec in TOOLS.values()
    ]
    detected_context = {
        "city_detected_by_parser": extract_city(question),
        "matrix_detected_by_parser": extract_first_matrix(question),
    }

    prompt = f"""
Select exactly one tool for the user question.

Available tools:
{json.dumps(tool_descriptions, indent=2)}

Detected context:
{json.dumps(detected_context, indent=2)}

Return exactly one JSON object and nothing else:
{{"tool": "get_weather", "arguments": {{"city": "London"}}, "reason": "weather question"}}
or
{{"tool": "invert_matrix", "arguments": {{"matrix": [[1,0],[0,1]], "decimals": 5}}, "reason": "matrix inversion question"}}
or
{{"tool": "search_web", "arguments": {{"query": "recent information about Agentic AI"}}, "reason": "information retrieval"}}
or
{{"tool":"search_web", "arguments":{{"query": "What is the capital of Namibia and its population"}}, "reason":"general knowledge"}}
or
{{"tool":"convert_currency", "arguments":{{"amount":100, "from_currency":"EUR", "to_currency":"USD"}}, "reason":"currency conversion"}}
or
{{"tool": "calculate", "arguments": {{"expression":"250*4"}}, "reason":"exchange rate already provided"}}
or
{{"tool": "calculate", "arguments": {{"expression":"250*4"}}, "reason":"exchange rate already provided"}}

Rules:
- For weather, use get_weather and include city.
- For matrix inversion, use invert_matrix and include the full matrix.
- For general knowledge, information lookup, news, facts, population, and similar questions, use search_web and include the query.
- Use convert_currency only when a live exchange rate is required.
- Use calculate when all required numbers are already present in the question.
- If the exchange rate is already given in the question, do NOT use convert_currency.
- For arithmetic using values already present in the question, use calculate.
- If the question contains variables (X, Y, unknown values), perform symbolic reasoning instead of currency conversion.
- Do not use weather unless explicit weather query.
- Do not calculate by yourself.
- Do not answer directly.

User question:
{question}
""".strip()

    raw = qwen_generate(prompt, max_new_tokens=150)
    #print(raw)
    parsed = parse_json_from_text(raw)
    normalized = normalize_model_tool_call(parsed, question, raw)
    if normalized is not None:
        return normalized

    repair_prompt = f"""
Your previous output was not a valid tool call.

Previous output:
{raw}

Available tool names:
{list(TOOLS.keys())}

Detected context:
{json.dumps(detected_context)}

Return only a valid JSON tool call:
{{"tool": "tool_name", "arguments": {{...}}, "reason": "short reason"}}

User question:
{question}
""".strip()

    repaired_raw = qwen_generate(repair_prompt, max_new_tokens=120)
    repaired = parse_json_from_text(repaired_raw)
    normalized = normalize_model_tool_call(repaired, question, repaired_raw)
    if normalized is not None:
        return normalized

    guarded = guarded_tool_call_from_question(question, raw + "\n--- repair attempt ---\n" + repaired_raw)
    if guarded is not None:
        return guarded

    return {
        "tool": "none",
        "arguments": {},
        "reason": "Qwen was invoked, but no valid safe tool call could be produced or repaired.",
        "source": "qwen_invalid",
        "model_raw": raw,
    }


def propose_tool_call(question: str) -> Dict[str, Any]:
    """Use the required Qwen model to propose a validated tool call."""
    return qwen_propose_tool_call(question)

In [96]:
@dataclass
class AgentMemory:
    observations: List[Dict[str, Any]] = field(default_factory=list)
    final_answers: List[str] = field(default_factory=list)

    def remember(self, item: Dict[str, Any]) -> None:
        self.observations.append(item)


class ToolUseAgent:
    def __init__(self, max_steps: int = 2):
        self.max_steps = max_steps
        self.memory = AgentMemory()

    def run(self, question: str) -> Dict[str, Any]:
        trace: List[Dict[str, Any]] = []

        for step in range(1, self.max_steps + 1):
            tool_call = propose_tool_call(question)
            trace.append({
                "step": step,
                "stage": "plan/router",
                "source": tool_call.get("source"),
                "tool": tool_call.get("tool"),
                "reason": tool_call.get("reason"),
                "arguments": tool_call.get("arguments"),
                "model_raw": tool_call.get("model_raw", ""),
            })

            if tool_call.get("tool") == "none":
                answer = "No safe registered tool call could be produced. Provide a city for weather or a numeric matrix for inversion."
                trace.append({"step": step, "stage": "final", "answer": answer})
                return {"answer": answer, "trace": pd.DataFrame(trace), "memory": self.memory}

            result = call_tool(tool_call["tool"], tool_call["arguments"])
            observation = {
                "tool": result.tool,
                "ok": result.ok,
                "data": result.data,
                "error": result.error,
                "latency_ms": round(result.latency_ms, 2),
            }
            self.memory.remember(observation)
            trace.append({"step": step, "stage": "observe", **observation})

            if result.ok:
                answer = self.format_answer(result)
                self.memory.final_answers.append(answer)
                trace.append({"step": step, "stage": "verify", "status": "success"})
                trace.append({"step": step, "stage": "final", "answer": answer})
                return {"answer": answer, "trace": pd.DataFrame(trace), "memory": self.memory}

            trace.append({"step": step, "stage": "verify", "status": "tool_error", "error": result.error})
            break

        answer = "The agent could not complete the task. Review the trace and adjust the input or tool policy."
        trace.append({"step": self.max_steps, "stage": "final", "answer": answer})
        return {"answer": answer, "trace": pd.DataFrame(trace), "memory": self.memory}

    @staticmethod
    def format_answer(result: ToolResult) -> str:
        if result.tool == "get_weather":
            d = result.data
            return (
                f"Current weather in {d['location']}: {d['temperature']}, "
                f"{d['condition']}, humidity {d['humidity']}, wind {d['wind']}."
            )

        if result.tool == "invert_matrix":
            d = result.data
            inv = np.array(d["inverse"])
            return (
                f"The {d['shape'][0]}x{d['shape'][1]} matrix is invertible. "
                f"det(A)={d['determinant']}, condition number={d['condition_number']}.\n\n"
                f"Inverse matrix:\n{inv}\n\n"
                f"Residual max error: {d['residual_max_error']:.2e}"
            )

        if result.tool == "search_web":
            d = result.data
            results = d.get("results", [])
            if not results:
                return "No search results found."
            search_context = "\n\n".join(
                [
                    f"Title: {r['title']}\n"
                    f"Content: {r['body']}\n"
                    f"Source: {r['href']}"
                    for r in results[:5]
                ])
            prompt = f"""
            Answer the user's question using only the search results below.

            Question:
            {d.get('query', '')}

            Search results:
            {search_context}

            Instructions:
            - Answer ONLY the question that was asked.
            - Give a concise, direct natural-language answer.
            - Summarize the RELEVANT information for the answer.
            - Do not list or summarize all search results.
            - Answer in 1-4 sentences - the less the better.
            - Do NOT return JSON.
            - Do NOT return dictionaries.
            - Do NOT use fields such as "answer", "tool", "arguments", or "reason".
            - If the answer is not available, say so.
            """
            return qwen_generate(prompt, max_new_tokens=200).strip()

        if result.tool == "convert_currency":
            d = result.data

            return (
                f"{d['amount']} {d['from']} = "
                f"{d['converted']} {d['to']} "
                f"(exchange rate {d['rate']:.4f})"
            )



        return str(result.data)

In [79]:

agent = ToolUseAgent(max_steps=2)
weather_result = agent.run("What is actual weather in Vienna?")

print(weather_result["answer"])
weather_result["trace"]

Current weather in Vienna, Austria: 29.6 °C, overcast, humidity 44 %, wind 12.1 km/h.


,step,stage,source,tool,reason,arguments,model_raw,ok,data,error,latency_ms,status,answer
0,1,plan/router,qwen,get_weather,weather question,{'city': 'Vienna'},"{""tool"": ""get_weather"", ""arguments"": {""city"": ""Vienna""}, ""reason"": ""weather question""}",NaN,NaN,NaN,NaN,NaN,NaN
1,1,observe,NaN,get_weather,NaN,NaN,NaN,True,"{'location': 'Vienna, Austria', 'temperature': '29.6 °C', 'humidity': '44 %', 'condition_code': 3, 'condition': 'overcast', 'wind': '12....",,536.1,NaN,NaN
2,1,verify,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,success,NaN
3,1,final,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Current weather in Vienna, Austria: 29.6 °C, overcast, humidity 44 %, wind 12.1 km/h."


In [80]:
matrix_4x4 = [
    [4, 7, 2, 3],
    [0, 5, 0, 1],
    [0, 0, 3, 0],
    [2, 0, 1, 6],
]

question_matrix = f"Solve matrix inversion for {matrix_4x4}."

agent = ToolUseAgent(max_steps=2)
matrix_result = agent.run(question_matrix)

print(matrix_result["answer"])
matrix_result["trace"]

The 4x4 matrix is invertible. det(A)=312.0, condition number=6.11902.

Inverse matrix:
[[ 0.28846 -0.40385 -0.16667 -0.07692]
 [ 0.01923  0.17308  0.      -0.03846]
 [ 0.       0.       0.33333  0.     ]
 [-0.09615  0.13462  0.       0.19231]]

Residual max error: 2.22e-16


,step,stage,source,tool,reason,arguments,model_raw,ok,data,error,latency_ms,status,answer
0,1,plan/router,qwen,invert_matrix,matrix inversion question,"{'matrix': [[4, 7, 2, 3], [0, 5, 0, 1], [0, 0, 3, 0], [2, 0, 1, 6]], 'decimals': 5}","{""tool"": ""invert_matrix"", ""arguments"": {""matrix"": [[4, 7, 2, 3], [0, 5, 0, 1], [0, 0, 3, 0], [2, 0, 1, 6]], ""decimals"": 5}, ""reason"": ""m...",NaN,NaN,NaN,NaN,NaN,NaN
1,1,observe,NaN,invert_matrix,NaN,NaN,NaN,True,"{'shape': [4, 4], 'determinant': 312.0, 'condition_number': 6.11902, 'inverse': [[0.28846, -0.40385, -0.16667, -0.07692], [0.01923, 0.17...",,0.23,NaN,NaN
2,1,verify,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,success,NaN
3,1,final,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"The 4x4 matrix is invertible. det(A)=312.0, condition number=6.11902.\n\nInverse matrix:\n[[ 0.28846 -0.40385 -0.16667 -0.07692]\n [ 0.0..."


In [97]:
agent = ToolUseAgent(max_steps=2)
q1_result = agent.run("What is the capital of Austria, and what is its current population?")

print(q1_result["answer"])
q1_result["trace"]

"Vienna is the capital of Austria and has a population of approximately 1.757 million."


,step,stage,source,tool,reason,arguments,model_raw,ok,data,error,latency_ms,status,answer
0,1,plan/router,qwen,search_web,Qwen selected this tool.,"{'query': 'capital of Austria and its current population', 'max_results': 5}","{""tool"": ""search_web"", ""arguments"": {""query"": ""capital of Austria and its current population""}}",NaN,NaN,NaN,NaN,NaN,NaN
1,1,observe,NaN,search_web,NaN,NaN,NaN,True,"{'query': 'capital of Austria and its current population', 'results': [{'title': 'Vienna - Wikipedia', 'href': 'https://en.wikipedia.org...",,1724.77,NaN,NaN
2,1,verify,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,success,NaN
3,1,final,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"""Vienna is the capital of Austria and has a population of approximately 1.757 million."""


In [98]:
agent = ToolUseAgent(max_steps=2)
q1_result = agent.run("Search for recent information about Agentic AI and summarize it in three sentences.")

print(q1_result["answer"])
q1_result["trace"]

Recent information indicates that Agentic AI is reshaping enterprise architecture by enabling agents to access diverse data sources like SharePoint documents and HR files, while survey data suggests a growing willingness among users to share personal information with AI agents.


,step,stage,source,tool,reason,arguments,model_raw,ok,data,error,latency_ms,status,answer
0,1,plan/router,qwen,search_web,information retrieval,"{'query': 'recent information about Agentic AI', 'max_results': 5}","{""tool"": ""search_web"", ""arguments"": {""query"": ""recent information about Agentic AI""}, ""reason"": ""information retrieval""}",NaN,NaN,NaN,NaN,NaN,NaN
1,1,observe,NaN,search_web,NaN,NaN,NaN,True,"{'query': 'recent information about Agentic AI', 'results': [{'title': 'Agentic AI is driving rethink of enterprise architecture and', '...",,2825.39,NaN,NaN
2,1,verify,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,success,NaN
3,1,final,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Recent information indicates that Agentic AI is reshaping enterprise architecture by enabling agents to access diverse data sources like...


In [99]:
agent = ToolUseAgent(max_steps=2)
q1_result = agent.run("Find information about MCP and explain why it is useful for agent tools.")

print(q1_result["answer"])
q1_result["trace"]

MCP and agent tools are distinct but complementary concepts in AI agent development. MCP is a standardized protocol from Anthropic that allows agents to access external tools and services via a standardized interface, while agent tools are the specific functions and services (like APIs or databases) that agents can call to interact with the world. Together, they enable agents to perform complex tasks by leveraging external resources through a unified protocol.


,step,stage,source,tool,reason,arguments,model_raw,ok,data,error,latency_ms,status,answer
0,1,plan/router,qwen,search_web,Qwen selected this tool.,"{'query': 'MCP and agent tools explanation', 'max_results': 5}","{""tool"": ""search_web"", ""arguments"": {""query"": ""MCP and agent tools explanation""}}",NaN,NaN,NaN,NaN,NaN,NaN
1,1,observe,NaN,search_web,NaN,NaN,NaN,True,"{'query': 'MCP and agent tools explanation', 'results': [{'title': 'Using MCP tools with Agents - Microsoft Learn', 'href': 'https://lea...",,1175.62,NaN,NaN
2,1,verify,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,success,NaN
3,1,final,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MCP and agent tools are distinct but complementary concepts in AI agent development. MCP is a standardized protocol from Anthropic that ...


In [100]:
agent = ToolUseAgent(max_steps=2)
q1_result = agent.run("What is Tesla's current stock price?")

print(q1_result["answer"])
q1_result["trace"]

Tesla stock price in 2024 was approximately $183.28, with a 52-week high of $498.83 and a 52-week low of $24.86, according to the provided price history data.


,step,stage,source,tool,reason,arguments,model_raw,ok,data,error,latency_ms,status,answer
0,1,plan/router,qwen,search_web,Qwen selected this tool.,"{'query': 'Tesla stock price 2024', 'max_results': 5}","{""tool"": ""search_web"", ""arguments"": {""query"": ""Tesla stock price 2024""}}",NaN,NaN,NaN,NaN,NaN,NaN
1,1,observe,NaN,search_web,NaN,NaN,NaN,True,"{'query': 'Tesla stock price 2024', 'results': [{'title': 'Tesla - 16 Year Stock Price History | TSLA - Macrotrends', 'href': 'https://w...",,480.69,NaN,NaN
2,1,verify,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,success,NaN
3,1,final,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Tesla stock price in 2024 was approximately $183.28, with a 52-week high of $498.83 and a 52-week low of $24.86, according to the provid..."


In [121]:
agent = ToolUseAgent(max_steps=2)
q1_result = agent.run("Convert 100 EUR to USD.")

print(q1_result["answer"])
q1_result["trace"]

100 EUR = 114.11 USD (exchange rate 1.1411)


,step,stage,source,tool,reason,arguments,model_raw,ok,data,error,latency_ms,status,answer
0,1,plan/router,qwen,convert_currency,currency conversion,"{'amount': 100, 'from_currency': 'EUR', 'to_currency': 'USD'}","{""tool"":""convert_currency"",""arguments"":{""amount"":100,""from_currency"":""EUR"",""to_currency"":""USD""},""reason"":""currency conversion""}",NaN,NaN,NaN,NaN,NaN,NaN
1,1,observe,NaN,convert_currency,NaN,NaN,NaN,True,"{'amount': 100, 'from': 'EUR', 'to': 'USD', 'rate': 1.1411, 'converted': 114.11}",,290.33,NaN,NaN
2,1,verify,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,success,NaN
3,1,final,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100 EUR = 114.11 USD (exchange rate 1.1411)


In [120]:
agent = ToolUseAgent(max_steps=2)
q1_result = agent.run("If 1 EUR equals X USD, how much is 250 EUR in USD?")

print(q1_result["answer"])
q1_result["trace"]

{'expression': '250*1', 'result': 250}


,step,stage,source,tool,reason,arguments,model_raw,ok,data,error,latency_ms,status,answer
0,1,plan/router,qwen,calculate,Qwen selected this tool.,{'expression': '250*1'},"{""tool"": ""calculate"", ""arguments"": {""expression"":""250*1""}}",NaN,NaN,NaN,NaN,NaN,NaN
1,1,observe,NaN,calculate,NaN,NaN,NaN,True,"{'expression': '250*1', 'result': 250}",,0.04,NaN,NaN
2,1,verify,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,success,NaN
3,1,final,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'expression': '250*1', 'result': 250}"


In [103]:
agent = ToolUseAgent(max_steps=2)
q1_result = agent.run("Compare EUR to GBP and USD using live exchange-rate data.")

print(q1_result["answer"])
q1_result["trace"]

100 EUR = 114.13 USD (exchange rate 1.1413)


,step,stage,source,tool,reason,arguments,model_raw,ok,data,error,latency_ms,status,answer
0,1,plan/router,qwen,convert_currency,currency conversion,"{'amount': 100, 'from_currency': 'EUR', 'to_currency': 'USD'}","{""tool"":""convert_currency"",""arguments"":{""amount"":100,""from_currency"":""EUR"",""to_currency"":""USD""},""reason"":""currency conversion""}",NaN,NaN,NaN,NaN,NaN,NaN
1,1,observe,NaN,convert_currency,NaN,NaN,NaN,True,"{'amount': 100, 'from': 'EUR', 'to': 'USD', 'rate': 1.1413, 'converted': 114.13}",,273.16,NaN,NaN
2,1,verify,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,success,NaN
3,1,final,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100 EUR = 114.13 USD (exchange rate 1.1413)


Convert 100 EUR to USD.

If 1 EUR equals X USD, how much is 250 EUR in USD?

Compare EUR to GBP and USD using live exchange-rate data.

In [86]:
def handle_mcp_request(request: Dict[str, Any]) -> Dict[str, Any]:
    """Handle a minimal MCP-style JSON-RPC request."""
    method = request.get("method")
    request_id = request.get("id")

    if method == "tools/list":
        return {"jsonrpc": "2.0", "id": request_id, "result": {"tools": mcp.list_tools()}}

    if method == "tools/call":
        params = request.get("params", {})
        name = params.get("name")
        arguments = params.get("arguments", {})
        result = mcp.call_tool(name, arguments)
        return {"jsonrpc": "2.0", "id": request_id, "result": result}

    return {"jsonrpc": "2.0", "id": request_id, "error": f"Unsupported method: {method}"}


handle_mcp_request({"jsonrpc": "2.0", "id": 1, "method": "tools/list"})

{'jsonrpc': '2.0',
 'id': 1,
 'result': {'tools': [{'name': 'get_weather',
    'description': 'Gets current weather for a city using Open-Meteo.\n\n    Args:\n        city: city name, for example "Vienna".\n\n    Returns:\n        A dictionary with location, temperature, humidity, condition_code, condition, and wind.'},
   {'name': 'invert_matrix',
    'description': 'Invert a square numeric matrix.\n\n    Args:\n        matrix: square matrix as a nested Python list.\n        decimals: number of decimals used for the displayed inverse.\n        condition_threshold: reject very ill-conditioned matrices.\n\n    Returns:\n        determinant, condition_number, inverse, and residual_max_error.'},
   {'name': 'search_web',
    'description': 'Search the web using DDGS.\n\n    Args:\n        query: search query\n        max_results: number of results\n\n    Returns:\n        Search results or error.'},
   {'name': 'convert_currency', 'description': ''},
   {'name': 'calculate', 'description'

In [87]:
weather_mcp_request = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {"name": "get_weather", "arguments": {"city": "Vienna"}},
}

handle_mcp_request(weather_mcp_request)

{'jsonrpc': '2.0',
 'id': 2,
 'result': {'location': 'Vienna, Austria',
  'temperature': '29.6 °C',
  'humidity': '44 %',
  'condition_code': 3,
  'condition': 'overcast',
  'wind': '12.1 km/h'}}

In [88]:
def evaluate_trace(trace_df: pd.DataFrame) -> Dict[str, Any]:
    if trace_df.empty:
        return {"success": 0, "tool_calls": 0, "tool_errors": 0, "final_answer": False}

    stage = trace_df.get("stage", pd.Series([""] * len(trace_df)))
    ok = trace_df.get("ok", pd.Series([None] * len(trace_df)))
    status = trace_df.get("status", pd.Series([""] * len(trace_df)))

    tool_calls = int((stage == "observe").sum())
    tool_errors = int(((stage == "observe") & (ok == False)).sum())
    success = int(((stage == "verify") & (status == "success")).any())
    final_answer = bool((stage == "final").any())

    return {
        "success": success,
        "tool_calls": tool_calls,
        "tool_errors": tool_errors,
        "final_answer": final_answer,
    }


print("Weather trace metrics:", evaluate_trace(weather_result["trace"]))
print("Matrix trace metrics:", evaluate_trace(matrix_result["trace"]))


Weather trace metrics: {'success': 1, 'tool_calls': 1, 'tool_errors': 0, 'final_answer': True}
Matrix trace metrics: {'success': 1, 'tool_calls': 1, 'tool_errors': 0, 'final_answer': True}


In [89]:
import sys
import unittest
from unittest.mock import Mock, patch


class AgenticAIToolTests(unittest.TestCase):
    def test_model_is_loaded(self):
        self.assertIsNotNone(tokenizer)
        self.assertIsNotNone(model)
        self.assertEqual(MODEL_ID, "Qwen/Qwen3.5-0.8B")

    def test_registered_tools(self):
        self.assertIn("get_weather", TOOLS)
        self.assertIn("invert_matrix", TOOLS)
        self.assertIn("get_weather", mcp.tools)
        self.assertIn("invert_matrix", mcp.tools)

    def test_matrix_inverse_4x4_residual(self):
        result = invert_matrix(matrix_4x4, decimals=8)
        self.assertNotIn("error", result)
        self.assertEqual(result["shape"], [4, 4])
        self.assertLess(result["residual_max_error"], 1e-9)

    def test_qwen_router_weather_json_contract(self):
        fake_qwen_json = '{"tool": "get_weather", "arguments": {"city": "Vienna"}, "reason": "current weather"}'
        with patch.object(sys.modules[__name__], "qwen_generate", return_value=fake_qwen_json):
            call = propose_tool_call("What is actual weather in Vienna?")
        self.assertEqual(call["source"], "qwen")
        self.assertEqual(call["tool"], "get_weather")
        self.assertEqual(call["arguments"]["city"], "Vienna")

    def test_qwen_router_matrix_json_contract(self):
        fake_qwen_json = json.dumps({
            "tool": "invert_matrix",
            "arguments": {"matrix": matrix_4x4, "decimals": 5},
            "reason": "matrix inversion",
        })
        q = f"Solve matrix inversion for {matrix_4x4}."
        with patch.object(sys.modules[__name__], "qwen_generate", return_value=fake_qwen_json):
            call = propose_tool_call(q)
        self.assertEqual(call["source"], "qwen")
        self.assertEqual(call["tool"], "invert_matrix")
        self.assertEqual(np.array(call["arguments"]["matrix"]).shape, (4, 4))

    def test_guardrail_repair_after_qwen_invocation(self):
        # Qwen is still invoked, but the verifier repairs the tool-call structure
        # for an unambiguous weather question.
        with patch.object(sys.modules[__name__], "qwen_generate", return_value="I would use the weather tool."):
            call = propose_tool_call("What is actual weather in Vienna?")
        self.assertEqual(call["source"], "qwen_guardrail_repair")
        self.assertEqual(call["tool"], "get_weather")
        self.assertEqual(call["arguments"]["city"], "Vienna")

    @patch("requests.get")
    def test_weather_tool_mocked(self, mock_get):
        geo_response = Mock()
        geo_response.raise_for_status.return_value = None
        geo_response.json.return_value = {
            "results": [{"name": "Vienna", "country": "Austria", "latitude": 48.2085, "longitude": 16.3721}]
        }

        weather_response = Mock()
        weather_response.raise_for_status.return_value = None
        weather_response.json.return_value = {
            "current": {
                "temperature_2m": 21.5,
                "relative_humidity_2m": 55,
                "weather_code": 2,
                "wind_speed_10m": 7.2,
            },
            "current_units": {
                "temperature_2m": "°C",
                "relative_humidity_2m": "%",
                "wind_speed_10m": "km/h",
            },
        }

        mock_get.side_effect = [geo_response, weather_response]
        result = get_weather("Vienna")
        self.assertEqual(result["location"], "Vienna, Austria")
        self.assertEqual(result["temperature"], "21.5 °C")
        self.assertEqual(result["condition"], "partly cloudy")

    def test_mcp_matrix_call(self):
        response = handle_mcp_request({
            "jsonrpc": "2.0",
            "id": 10,
            "method": "tools/call",
            "params": {"name": "invert_matrix", "arguments": {"matrix": matrix_4x4, "decimals": 5}},
        })
        self.assertEqual(response["id"], 10)
        self.assertIn("inverse", response["result"])


suite = unittest.defaultTestLoader.loadTestsFromTestCase(AgenticAIToolTests)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)


test_guardrail_repair_after_qwen_invocation (__main__.AgenticAIToolTests.test_guardrail_repair_after_qwen_invocation) ... ok
test_matrix_inverse_4x4_residual (__main__.AgenticAIToolTests.test_matrix_inverse_4x4_residual) ... ok
test_mcp_matrix_call (__main__.AgenticAIToolTests.test_mcp_matrix_call) ... ok
test_model_is_loaded (__main__.AgenticAIToolTests.test_model_is_loaded) ... ok
test_qwen_router_matrix_json_contract (__main__.AgenticAIToolTests.test_qwen_router_matrix_json_contract) ... ok
test_qwen_router_weather_json_contract (__main__.AgenticAIToolTests.test_qwen_router_weather_json_contract) ... ok
test_registered_tools (__main__.AgenticAIToolTests.test_registered_tools) ... ok
test_weather_tool_mocked (__main__.AgenticAIToolTests.test_weather_tool_mocked) ... ok

----------------------------------------------------------------------
Ran 8 tests in 0.007s

OK


<unittest.runner.TextTestResult run=8 errors=0 failures=0>

In [90]:
def diagnose_trace(trace_df: pd.DataFrame) -> List[str]:
    diagnosis = []
    if trace_df.empty:
        return ["No trace was produced. Add logging before deployment."]

    stage = trace_df.get("stage", pd.Series([""] * len(trace_df)))
    ok = trace_df.get("ok", pd.Series([None] * len(trace_df)))
    status = trace_df.get("status", pd.Series([""] * len(trace_df)))

    if not (stage == "observe").any():
        diagnosis.append("No tool was called. Check planner/router output and required arguments.")

    if ((stage == "observe") & (ok == False)).any():
        diagnosis.append("A tool returned an error. Check arguments, network access, or tool recovery policy.")

    if not ((stage == "verify") & (status == "success")).any():
        diagnosis.append("No successful verification step. Add or fix verification before final answers.")

    if not diagnosis:
        diagnosis.append("Trace looks complete: tool call, observation, verification, and final answer exist.")

    return diagnosis


print("Weather diagnosis:")
for item in diagnose_trace(weather_result["trace"]):
    print("-", item)

print("\nMatrix diagnosis:")
for item in diagnose_trace(matrix_result["trace"]):
    print("-", item)

Weather diagnosis:
- Trace looks complete: tool call, observation, verification, and final answer exist.

Matrix diagnosis:
- Trace looks complete: tool call, observation, verification, and final answer exist.
